# CodeBERT experiments in Google Colab
Run with a GPU runtime. This notebook trains the synthetic eight-class and CodeContests binary experiments and copies outputs to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil
import subprocess
import zipfile

PROJECT_ZIP = Path('/content/drive/MyDrive/error-pattern-project.zip')
PROJECT_DIR = Path('/content/project')
REPO_URL = 'https://github.com/burymewithmykatana/programming-error-pattern-recognition.git'
BRANCH = 'codex/presentation-ready'

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if PROJECT_ZIP.exists():
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PROJECT_ZIP) as archive:
        archive.extractall(PROJECT_DIR)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

%cd /content/project
!pip install -q -e '.[transformer,datasets]'

Place the prepared split folders in `/content/drive/MyDrive/error-pattern-data/`. Use the reduced row limits below if runtime is constrained.

In [ ]:
from pathlib import Path
import zipfile

DATA_ROOT = Path('/content/drive/MyDrive/error-pattern-data')
DATA_ZIP = Path('/content/drive/MyDrive/error-pattern-data.zip')
OUTPUT = Path('/content/drive/MyDrive/error-pattern-results')
OUTPUT.mkdir(parents=True, exist_ok=True)

def has_required_splits(path):
    return all((path / relative).exists() for relative in [
        'synthetic/train.csv',
        'synthetic/test.csv',
        'code_contests/train.csv',
        'code_contests/test.csv',
    ])

def find_data_root(path):
    candidates = [path, path / 'error-pattern-data']
    if path.exists():
        candidates.extend(marker.parent.parent for marker in path.rglob('synthetic/train.csv'))
    for candidate in candidates:
        if has_required_splits(candidate):
            return candidate
    raise FileNotFoundError(
        'Could not find prepared CSV splits. Upload error-pattern-data.zip to '
        '/content/drive/MyDrive or place synthetic/ and code_contests/ under '
        '/content/drive/MyDrive/error-pattern-data.'
    )

if DATA_ZIP.exists() and not has_required_splits(DATA_ROOT):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as archive:
        archive.extractall(DATA_ROOT)

DATA = find_data_root(DATA_ROOT)
print(f'Using data root: {DATA}')

In [ ]:
import subprocess

subprocess.run([
    'python', 'scripts/run_experiment.py',
    '--train-data', str(DATA / 'synthetic/train.csv'),
    '--test-data', str(DATA / 'synthetic/test.csv'),
    '--model-type', 'codebert',
    '--config', 'configs/deep_model.yaml',
    '--output', str(OUTPUT / 'synthetic_codebert'),
], check=True)

In [ ]:
subprocess.run([
    'python', 'scripts/run_experiment.py',
    '--train-data', str(DATA / 'code_contests/train.csv'),
    '--test-data', str(DATA / 'code_contests/test.csv'),
    '--model-type', 'codebert',
    '--config', 'configs/deep_model.yaml',
    '--output', str(OUTPUT / 'codecontests_codebert'),
], check=True)

## Reduced-data fallback
If Colab runs out of memory or time, sample each training class to 2,000 rows, keep the official test set unchanged, and rerun. Record the reduced row count in the report.

In [ ]:
import pandas as pd
def reduced_train(source, destination, rows_per_class=2000):
    frame = pd.read_csv(source)
    reduced = frame.groupby('label', group_keys=False).apply(
        lambda group: group.sample(min(len(group), rows_per_class), random_state=42),
        include_groups=False,
    ).reset_index(drop=True)
    reduced.to_csv(destination, index=False)
    return reduced['label'].value_counts()
# Example:
# reduced_train(DATA/'code_contests/train.csv', '/content/codecontests_reduced.csv')